# V3-1 — Evaluation protocol and cross-validation folds

This notebook freezes the V3 evaluation protocol. It preserves the existing 480/120 development split and creates five stratified outer folds for final model comparisons. No model is trained here.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from math import comb
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

DATA_ROOT = Path(r'P:\NexarCollisionData')
MANIFEST_ROOT = DATA_ROOT / 'manifests_v3'
REPORT_ROOT = DATA_ROOT / 'reports_v3'
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

VIDEO_MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
DEVELOPMENT_SPLIT_PATH = DATA_ROOT / 'metadata_split_v1.csv'
RAW_TRAIN_METADATA_PATH = DATA_ROOT / 'train_metadata.csv'
REGISTRY_PATH = REPORT_ROOT / 'experiments_v3_registry.csv'

N_SPLITS = 5
SEED = 42
INNER_VALIDATION_FRACTION = 0.10
MINIMUM_ACCIDENT_RECALL = 0.85
BOOTSTRAP_SAMPLES = 2000

assert VIDEO_MANIFEST_PATH.is_file()
assert DEVELOPMENT_SPLIT_PATH.is_file()
assert RAW_TRAIN_METADATA_PATH.is_file()
print({'data_root': str(DATA_ROOT), 'n_splits': N_SPLITS, 'seed': SEED, 'minimum_accident_recall': MINIMUM_ACCIDENT_RECALL})


{'data_root': 'P:\\NexarCollisionData', 'n_splits': 5, 'seed': 42, 'minimum_accident_recall': 0.85}


In [2]:
# 1) Validate the frozen development split and choose the valid CV strategy
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

video_manifest = pd.read_csv(VIDEO_MANIFEST_PATH)
development_split = pd.read_csv(DEVELOPMENT_SPLIT_PATH)
raw_metadata = pd.read_csv(RAW_TRAIN_METADATA_PATH)
for table in [video_manifest, development_split]:
    table['video_id'] = table['video_id'].astype(str)
    table['label'] = table['label'].astype(int)

assert len(video_manifest) == 600
assert video_manifest['video_id'].is_unique
assert len(development_split) == 600
assert development_split['video_id'].is_unique
assert set(video_manifest['video_id']) == set(development_split['video_id'])
manifest_split = video_manifest.set_index('video_id')['split'].reindex(development_split['video_id']).to_numpy()
assert np.array_equal(manifest_split, development_split['split'].to_numpy())
assert pd.crosstab(development_split['split'], development_split['label']).to_dict('index') == {'train': {0: 240, 1: 240}, 'validation': {0: 60, 1: 60}}

candidate_group_columns = ['trip_id', 'driver_id', 'camera_id', 'route_id']
available_group_columns = [column for column in candidate_group_columns if column in video_manifest.columns or column in raw_metadata.columns]
assert not available_group_columns, 'A real group column was found. Stop and switch to StratifiedGroupKFold.'
cv_strategy = 'StratifiedKFold_at_video_id_level'
print({'cv_strategy': cv_strategy, 'available_group_columns': available_group_columns})
display(pd.crosstab(development_split['split'], development_split['label']))


{'cv_strategy': 'StratifiedKFold_at_video_id_level', 'available_group_columns': []}


label,0,1
split,,
train,240,240
validation,60,60


In [3]:
# 2) Create and freeze five outer folds
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_id = np.full(len(video_manifest), -1, dtype=int)
for fold, (_, validation_indices) in enumerate(splitter.split(video_manifest['video_id'], video_manifest['label'])):
    fold_id[validation_indices] = fold

assert (fold_id >= 0).all()
cv_folds = video_manifest[['video_id', 'label', 'split']].rename(columns={'split': 'development_split'}).copy()
cv_folds['outer_fold'] = fold_id
cv_folds['cv_strategy'] = cv_strategy
cv_folds['group_column'] = ''
cv_folds['fold_seed'] = SEED
cv_folds['source_video_manifest_sha256'] = sha256_file(VIDEO_MANIFEST_PATH)
cv_folds['role_when_outer_fold_is_active'] = 'validation_in_its_outer_fold'

fold_class_counts = cv_folds.groupby(['outer_fold', 'label']).size().unstack(fill_value=0)
assert cv_folds['video_id'].is_unique
assert fold_class_counts.shape == (N_SPLITS, 2)
assert fold_class_counts[0].eq(60).all() and fold_class_counts[1].eq(60).all()

CV_FOLDS_PATH = MANIFEST_ROOT / 'cv_folds_v3.csv'
cv_folds.sort_values(['outer_fold', 'label', 'video_id']).to_csv(CV_FOLDS_PATH, index=False)
fold_summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'videos': int(len(cv_folds)),
    'n_splits': N_SPLITS,
    'strategy': cv_strategy,
    'group_column': None,
    'seed': SEED,
    'videos_per_validation_fold': 120,
    'class_counts_by_fold': {str(index): {str(label): int(value) for label, value in row.items()} for index, row in fold_class_counts.iterrows()},
    'source_video_manifest_sha256': sha256_file(VIDEO_MANIFEST_PATH)
}
CV_SUMMARY_PATH = MANIFEST_ROOT / 'cv_folds_v3_summary.json'
CV_SUMMARY_PATH.write_text(json.dumps(fold_summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(fold_class_counts)
print('CV folds:', CV_FOLDS_PATH)
print('CV summary:', CV_SUMMARY_PATH)


label,0,1
outer_fold,,
0,60,60
1,60,60
2,60,60
3,60,60
4,60,60


CV folds: P:\NexarCollisionData\manifests_v3\cv_folds_v3.csv
CV summary: P:\NexarCollisionData\manifests_v3\cv_folds_v3_summary.json


In [4]:
# 3) Reusable video-level metrics, threshold selection, bootstrap CI and McNemar test
def binary_metrics(y_true, probabilities, threshold):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist()
    }

def choose_threshold(y_true, probabilities, minimum_recall=MINIMUM_ACCIDENT_RECALL):
    table = pd.DataFrame([binary_metrics(y_true, probabilities, float(value)) for value in np.round(np.arange(0.10, 0.901, 0.01), 2)])
    eligible = table.loc[table['recall'].ge(minimum_recall)]
    pool = eligible if len(eligible) else table
    selected = pool.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
    return float(selected['threshold']), table

def bootstrap_metric_ci(y_true, probabilities, threshold, metric_name, samples=BOOTSTRAP_SAMPLES, seed=SEED):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    rng = np.random.default_rng(seed)
    positive_indices = np.flatnonzero(y_true == 1)
    negative_indices = np.flatnonzero(y_true == 0)
    values = []
    for _ in range(samples):
        indices = np.concatenate([rng.choice(positive_indices, size=len(positive_indices), replace=True), rng.choice(negative_indices, size=len(negative_indices), replace=True)])
        values.append(binary_metrics(y_true[indices], probabilities[indices], threshold)[metric_name])
    values = np.asarray(values, dtype=float)
    return {'metric': metric_name, 'samples': int(samples), 'mean': float(values.mean()), 'ci_lower': float(np.quantile(values, 0.025)), 'ci_upper': float(np.quantile(values, 0.975))}

def exact_mcnemar_pvalue(y_true, prediction_a, prediction_b):
    y_true = np.asarray(y_true, dtype=int)
    correct_a = np.asarray(prediction_a, dtype=int) == y_true
    correct_b = np.asarray(prediction_b, dtype=int) == y_true
    only_a_correct = int(np.logical_and(correct_a, ~correct_b).sum())
    only_b_correct = int(np.logical_and(~correct_a, correct_b).sum())
    discordant = only_a_correct + only_b_correct
    if discordant == 0:
        p_value = 1.0
    else:
        smaller = min(only_a_correct, only_b_correct)
        p_value = min(1.0, 2.0 * sum(comb(discordant, value) for value in range(smaller + 1)) / (2 ** discordant))
    return {'only_a_correct': only_a_correct, 'only_b_correct': only_b_correct, 'discordant': discordant, 'p_value': float(p_value)}


In [5]:
# 4) Deterministic tests for the evaluation utilities
test_labels = np.array([0, 0, 0, 1, 1, 1])
test_probabilities = np.array([0.05, 0.25, 0.35, 0.45, 0.75, 0.95])
test_threshold, test_threshold_table = choose_threshold(test_labels, test_probabilities, minimum_recall=0.85)
test_metrics = binary_metrics(test_labels, test_probabilities, test_threshold)
test_ci = bootstrap_metric_ci(test_labels, test_probabilities, test_threshold, 'f1', samples=100, seed=SEED)
test_mcnemar = exact_mcnemar_pvalue(test_labels, np.array([0, 0, 0, 1, 1, 1]), np.array([0, 1, 0, 1, 0, 1]))
assert test_metrics['recall'] >= 0.85
assert 0.0 <= test_ci['ci_lower'] <= test_ci['ci_upper'] <= 1.0
assert 0.0 <= test_mcnemar['p_value'] <= 1.0
METRIC_TEST_PATH = REPORT_ROOT / 'metric_functions_test_v3.json'
METRIC_TEST_PATH.write_text(json.dumps({'status': 'pass', 'selected_threshold': test_threshold, 'metrics': test_metrics, 'bootstrap_ci': test_ci, 'mcnemar': test_mcnemar}, ensure_ascii=False, indent=2), encoding='utf-8')
print(METRIC_TEST_PATH.read_text(encoding='utf-8'))


{
  "status": "pass",
  "selected_threshold": 0.36,
  "metrics": {
    "threshold": 0.36,
    "accuracy": 1.0,
    "precision": 1.0,
    "recall": 1.0,
    "f1": 1.0,
    "roc_auc": 1.0,
    "pr_auc": 1.0,
    "confusion_matrix": [
      [
        3,
        0
      ],
      [
        0,
        3
      ]
    ]
  },
  "bootstrap_ci": {
    "metric": "f1",
    "samples": 100,
    "mean": 1.0,
    "ci_lower": 1.0,
    "ci_upper": 1.0
  },
  "mcnemar": {
    "only_a_correct": 2,
    "only_b_correct": 0,
    "discordant": 2,
    "p_value": 0.5
  }
}


In [6]:
# 5) Freeze the written protocol and result templates for future finalist models
PROTOCOL_PATH = REPORT_ROOT / 'evaluation_protocol_v3.md'
protocol_lines = [
    '# V3 evaluation protocol',
    '',
    'Generated: {}'.format(datetime.now(timezone.utc).isoformat()),
    '',
    '## Two evaluation levels',
    '- Development: the existing fixed 480 train / 120 validation split. It is used for fast, documented ablations only.',
    '- Final comparison: five outer StratifiedKFold folds over the 600 videos. Each outer validation fold has 120 videos: 60 accident and 60 non-accident.',
    '- No trip, driver, camera or route identifier exists in the available metadata. GroupKFold is not invented; the strategy is therefore stratified video-level K-fold.',
    '',
    '## Per-fold rules',
    '- Train on the 480 outer-train videos. Use a stratified inner validation subset of approximately 10 percent of those training videos for early stopping/checkpoint selection.',
    '- Do not use the outer validation fold for augmentation, early stopping, checkpoint selection or iterative hyperparameter tuning.',
    '- Save exactly one out-of-fold video probability for every video.',
    '- Select the final threshold only from all saved OOF predictions after the model configuration has been frozen.',
    '',
    '## Model-selection criteria',
    '- Primary: Accident F1 at MP4/video level.',
    '- Safety constraint: Accident Recall >= {:.2f}.'.format(MINIMUM_ACCIDENT_RECALL),
    '- Secondary: PR-AUC, ROC-AUC, precision, calibration, inference time and complementary FP/FN errors.',
    '',
    '## Statistics',
    '- Stratified video-level bootstrap with {} resamples for 95 percent CI of F1, Recall and PR-AUC.'.format(BOOTSTRAP_SAMPLES),
    '- Paired bootstrap and exact McNemar test for final model comparisons.',
    '',
    '## Frozen inputs',
    '- video manifest SHA-256: {}'.format(sha256_file(VIDEO_MANIFEST_PATH)),
    '- development split SHA-256: {}'.format(sha256_file(DEVELOPMENT_SPLIT_PATH)),
    '- CV fold manifest: {}'.format(CV_FOLDS_PATH)
]
PROTOCOL_PATH.write_text('\n'.join(protocol_lines) + '\n', encoding='utf-8')

BOOTSTRAP_TEMPLATE_PATH = REPORT_ROOT / 'bootstrap_results_v3.csv'
pd.DataFrame([{'model_id': '', 'evaluation_scope': 'pending_oof_predictions', 'metric': '', 'mean': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan, 'bootstrap_samples': BOOTSTRAP_SAMPLES, 'status': 'pending'}]).to_csv(BOOTSTRAP_TEMPLATE_PATH, index=False)

registry = pd.read_csv(REGISTRY_PATH)
protocol_row = {'run_id': 'V3_01_EVALUATION_PROTOCOL', 'stage': 'V3-1 cross_validation_protocol', 'model_id': 'none', 'dataset_version': 'v2_frozen_reference', 'split_version': 'cv_folds_v3', 'window_version': 'not_applicable', 'feature_version': 'not_applicable', 'augmentation_version': 'none', 'checkpoint_path': '', 'config_path': 'notebooks/31_v3_evaluation_protocol.ipynb', 'git_commit': 'not_available', 'status': 'completed', 'primary_metric': 'protocol_tests', 'primary_value': 1.0, 'notes': 'Five balanced video-level folds. Group fields absent; StratifiedKFold selected.'}
registry = registry.loc[~registry['run_id'].eq('V3_01_EVALUATION_PROTOCOL')]
registry = pd.concat([registry, pd.DataFrame([protocol_row])], ignore_index=True)
registry.to_csv(REGISTRY_PATH, index=False)

print(PROTOCOL_PATH.read_text(encoding='utf-8'))
print('Bootstrap template:', BOOTSTRAP_TEMPLATE_PATH)


# V3 evaluation protocol

Generated: 2026-08-02T04:59:08.580882+00:00

## Two evaluation levels
- Development: the existing fixed 480 train / 120 validation split. It is used for fast, documented ablations only.
- Final comparison: five outer StratifiedKFold folds over the 600 videos. Each outer validation fold has 120 videos: 60 accident and 60 non-accident.
- No trip, driver, camera or route identifier exists in the available metadata. GroupKFold is not invented; the strategy is therefore stratified video-level K-fold.

## Per-fold rules
- Train on the 480 outer-train videos. Use a stratified inner validation subset of approximately 10 percent of those training videos for early stopping/checkpoint selection.
- Do not use the outer validation fold for augmentation, early stopping, checkpoint selection or iterative hyperparameter tuning.
- Save exactly one out-of-fold video probability for every video.
- Select the final threshold only from all saved OOF predictions after the model con